# ⛽ 01 ETL: UK Fuel Prices

**Goal:** load, clean and join the two raw files, then save tidy files for the analysis and Power BI.

- 📥 **Extract:** read `price_history.csv` and `stations.csv`
- 🧹 **Transform:** remove bad rows, fix dates, join the files
- 💾 **Load:** save the result to `data/cleaned/v1/`

## ⚖️ Ethics note (ETL)

- 🔒 **Personal data:** none found. Columns are public station details (name, address, postcode) and fuel prices. Small independent stations could be sole traders, so the risk is low but not zero. Station names and postcodes are not used in any analysis or chart
- 📜 **Licence:** the dataset licence must be checked and credited (see README)
- 🧮 **Assumption:** prices under 100p or over 300p per litre are treated as errors (1,184 rows removed). This is a judgement call and is documented here
- ❓ **Missing values:** brands and cities are labelled "Unknown", not guessed
- 🗂️ **Governance:** raw files are never edited. Cleaned data is saved separately in `data/cleaned/v1/`

## 1️⃣ Import libraries

In [1]:
# pandas = tables, numpy = numbers
import pandas as pd
import numpy as np

## 2️⃣ Extract: load the raw data

In [2]:
# Notebook lives in jupyter_notebooks/, so we go up one folder (..) to reach data/
prices = pd.read_csv("../data/raw/price_history.csv")
stations = pd.read_csv("../data/raw/stations.csv")

print("Prices:  ", prices.shape)
print("Stations:", stations.shape)

Prices:   (1182085, 6)
Stations: (8503, 21)


In [3]:
prices.head()

,id,node_id,fuel_type,price_pence,recorded_at,source_updated_at
0,1,4882e3fee979cfefa29a8777ce57ca0ecea27b4f12ba7f...,E10,126.0,2026-02-07T21:40:01.511Z,2026-02-06T12:46:05.000Z
1,2,4882e3fee979cfefa29a8777ce57ca0ecea27b4f12ba7f...,B7_STANDARD,135.0,2026-02-07T21:40:01.511Z,2026-02-06T12:46:05.000Z
2,3,b739362af81acc9fec9eda6f155348125fa2d5c1772c96...,E10,127.0,2026-02-07T21:40:01.511Z,2026-02-01T11:04:45.000Z
3,4,b739362af81acc9fec9eda6f155348125fa2d5c1772c96...,B7_STANDARD,138.0,2026-02-07T21:40:01.511Z,2026-02-01T11:04:45.000Z
4,5,ae5b2f8f6979b8b2460ff52f85619dd0dfc6a10bc98ca1...,E5,146.0,2026-02-07T21:40:01.511Z,2026-02-06T13:57:50.000Z


In [4]:
stations.head()

,id,node_id,trading_name,brand_name,organisation_name,latitude,longitude,address_line_1,city,county,...,is_motorway,is_supermarket,is_temporarily_closed,is_permanently_closed,amenities,opening_times,fuel_types,first_seen,last_seen,updated_at
0,8502,805325cd44b7516728eb6fd6a2fe61da5d65d18a1ede52...,Garej Ni,Dragon,NaN,52.860600,-4.616600,Garej Ni,Pwllheli,NaN,...,False,False,False,False,"[""adblue_packaged"",""air_pump_or_screenwash"",""c...","{""usual_days"":{""friday"":{""open"":""07:00:00"",""cl...","[""B7_STANDARD"",""E10""]",2026-10-21T00:01:27.596Z,2026-10-21T00:01:27.260Z,2026-10-21T00:01:27.260Z
1,8496,613eb507f947e843b7474620e099da35200e1e1005970f...,Gotherington Cross Service Station,Murco,NaN,51.880000,2.080000,Murco,Bishop Cleeve,NaN,...,False,True,False,False,[],"{""usual_days"":{""friday"":{""open"":""06:00:00"",""cl...","[""B10"",""E10"",""E5""]",2026-10-13T00:01:33.422Z,2026-10-21T00:01:27.260Z,2026-10-13T00:01:32.927Z
2,8494,a457e8f05fd3b83e0fface7b9d3de91f99c66cf886be1d...,Welford Garage,Murco,NaN,52.169015,-1.784692,"WELFORD GARAGE, BINTON ROAD, WELFORD ON AVON, ...",STRATFORD-UPON-AVON,NaN,...,False,False,False,False,[],"{""usual_days"":{""friday"":{""open"":""08:00:00"",""cl...","[""B7_STANDARD"",""E5""]",2026-10-09T00:01:20.887Z,2026-10-21T00:01:27.260Z,2026-10-09T00:01:20.346Z
3,8501,7c00b1b4f6a411d721892b6899e532e2591ed606155381...,universal service stations ltd,Gulf,NaN,51.826665,-3.096528,"GILWERN GARAGE, CRICKHOWELL ROAD, GILWERN, Y F...",Y FENNI,NaN,...,False,True,False,False,"[""adblue_packaged"",""air_pump_or_screenwash"",""c...","{""usual_days"":{""friday"":{""open"":""07:00:00"",""cl...","[""B7_STANDARD"",""E10"",""E5""]",2026-10-20T00:01:10.986Z,2026-10-21T00:01:27.260Z,2026-10-20T00:01:10.431Z
4,8484,5981d0a8ef630907ef706b1c78b5115b60c633ae976060...,Braemar Filling Station,Gleaner,NaN,57.009600,-3.397660,Ballater Road,Aberdeenshire,NaN,...,False,False,False,False,"[""adblue_packaged"",""air_pump_or_screenwash"",""c...","{""usual_days"":{""friday"":{""open"":""08:00:00"",""cl...","[""B7_STANDARD"",""E10""]",2026-10-06T00:01:06.869Z,2026-10-21T00:01:27.260Z,2026-10-06T00:01:06.252Z


## 3️⃣ Inspect: check quality

In [5]:
# Missing values and fuel types
print("Missing values in prices:")
print(prices.isna().sum())
print("\nFuel types:")
print(prices["fuel_type"].value_counts())
print("\nPrice (pence) summary:")
print(prices["price_pence"].describe())

Missing values in prices:
id                   0
node_id              0
fuel_type            0
price_pence          0
recorded_at          0
source_updated_at    0
dtype: int64

Fuel types:


fuel_type
B7_STANDARD    439446
E10            356568
E5             209356
B7_PREMIUM     173648
HVO              1557
B10              1510
Name: count, dtype: int64

Price (pence) summary:
count    1.182085e+06
mean     1.730635e+02
std      2.944655e+01
min      0.000000e+00
25%      1.579000e+02
50%      1.729000e+02
75%      1.869000e+02
max      2.009900e+03
Name: price_pence, dtype: float64


In [6]:
# Missing values in stations (only columns that have some)
missing = stations.isna().sum()
missing[missing > 0]

brand_name             18
organisation_name    8306
city                  254
county               4705
dtype: int64

**What I found**
- 💷 Some prices are impossible (0 pence or 2,000+ pence), likely typing errors
- 🕳️ `organisation_name` and `county` are mostly empty
- ⛽ 6 fuel types, and 4 of them (E10, E5, B7 Standard, B7 Premium) make up almost everything

## 4️⃣ Transform: clean the prices

In [7]:
# Keep only realistic prices: 100 to 300 pence per litre
before = len(prices)
prices = prices[(prices["price_pence"] >= 100) & (prices["price_pence"] <= 300)].copy()
print(f"Removed {before - len(prices):,} impossible prices")

# Turn text dates into real dates
prices["recorded_at"] = pd.to_datetime(prices["recorded_at"], utc=True)

# Remove exact duplicates (same station, fuel and time)
before = len(prices)
prices = prices.drop_duplicates(subset=["node_id", "fuel_type", "recorded_at"])
print(f"Removed {before - len(prices):,} duplicates")

Removed 1,184 impossible prices


Removed 0 duplicates


In [8]:
# Group fuel types into simple labels (easier to read in charts)
fuel_groups = {
    "E10": "Petrol", "E5": "Petrol",
    "B7_STANDARD": "Diesel", "B7_PREMIUM": "Diesel",
}
prices["fuel_group"] = prices["fuel_type"].map(fuel_groups).fillna("Other")

# Simple date columns for time charts
prices["date"] = prices["recorded_at"].dt.date
prices["month"] = prices["recorded_at"].dt.strftime("%Y-%m")
prices["weekday"] = prices["recorded_at"].dt.day_name()

prices["fuel_group"].value_counts()

fuel_group
Diesel    612633
Petrol    565281
Other       2987
Name: count, dtype: int64

## 5️⃣ Transform: clean the stations

In [9]:
# Drop columns we do not need (empty or not useful for our questions)
stations = stations.drop(columns=["organisation_name", "county", "opening_times", "amenities"])

# Remove permanently closed stations
stations = stations[stations["is_permanently_closed"] == False]

# Fill the few missing brands and cities so charts do not break
stations["brand_name"] = stations["brand_name"].fillna("Unknown")
stations["city"] = stations["city"].fillna("Unknown").str.strip().str.title()   # BELFAST and Belfast become one city

# Station type label: Motorway / Supermarket / Other
stations["station_type"] = np.select(
    [stations["is_motorway"], stations["is_supermarket"]],
    ["Motorway", "Supermarket"],
    default="Other",
)

# Keep only the columns we need
stations = stations[["node_id", "trading_name", "brand_name", "city", "postcode",
                     "latitude", "longitude", "station_type", "is_temporarily_closed"]]
stations.shape

(8503, 9)

## 6️⃣ Join prices and stations

In [10]:
# Left join: keep every price, add station details using node_id
df = prices.merge(stations, on="node_id", how="inner")
print("Rows after join:", df.shape[0])
print("Stations with prices:", df["node_id"].nunique())

Rows after join: 1180846
Stations with prices: 8278


In [11]:
# Final check
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1180846 entries, 0 to 1180845
Data columns (total 18 columns):
 #   Column                 Non-Null Count    Dtype              
---  ------                 --------------    -----              
 0   id                     1180846 non-null  int64              
 1   node_id                1180846 non-null  str                
 2   fuel_type              1180846 non-null  str                
 3   price_pence            1180846 non-null  float64            
 4   recorded_at            1180846 non-null  datetime64[us, UTC]
 5   source_updated_at      1180846 non-null  str                
 6   fuel_group             1180846 non-null  str                
 7   date                   1180846 non-null  object             
 8   month                  1180846 non-null  str                
 9   weekday                1180846 non-null  str                
 10  trading_name           1180846 non-null  str                
 11  brand_name             1180846 non-

## 7️⃣ Load: save the cleaned data (version 1)

In [12]:
# Keep only the columns Power BI needs, so the file stays small (GitHub limit is 100 MB)
# Prices and stations stay as TWO tables linked by node_id (a clean Power BI model)
prices_slim = prices[["node_id", "fuel_type", "fuel_group", "price_pence", "recorded_at"]].copy()
prices_slim["recorded_at"] = prices_slim["recorded_at"].dt.strftime("%Y-%m-%d %H:%M")

# Saved in a versioned folder (v1) so changes can be tracked
prices_slim.to_csv("../data/cleaned/v1/prices_cleaned.csv", index=False)
stations.to_csv("../data/cleaned/v1/stations_cleaned.csv", index=False)
print("Saved to data/cleaned/v1/")

Saved to data/cleaned/v1/


## ✅ Summary
- 🧹 Removed impossible prices and duplicates
- 🔗 Joined prices with station details (brand, city, type)
- ⛽ Added `fuel_group`, `date`, `month`, `weekday`
- 💾 Saved `prices_cleaned.csv` and `stations_cleaned.csv` in `data/cleaned/v1/` (link them on `node_id` in Power BI)
- ➡️ Next: `02_Analysis_Visualization.ipynb`